<a href="https://colab.research.google.com/github/avikumart/DA-DS-Questions/blob/main/Python/Agentic_coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import asyncio, time, collections

In [3]:
# rate limiter that caps the LLM API called to N per second across the concurrent requests
class RateLimiter:
  def __init__(self, calls_per_sec: int):
    self.limit = calls_per_sec
    self.calls = collections.deque(maxlen=calls_per_sec)
    self._lock = asyncio.Lock()

  async def acquire(self):
    async with self._lock:
      now = time.monotonic()
      while self.calls and now - self.calls[0] >= 1.0:
        self.calls.popleft()
      if len(self.calls) >= self.limit:
        wait = (self.calls[0] + 1.0) - now
        await asyncio.sleep(max(wait, 0))
      self.calls.append(now)


limiter = RateLimiter(10)


async def safe_llm_call(prompt):
  await limiter.acquire()
  return await cerebras_client.chat(prompt)

In [6]:
# give the stream of the token outputs, count the frequency and return top-K most common tokens.
tokens = ["the","cat","sat","on","the","mat","the","cat"]

from collections import Counter
import heapq

def top_k_tokens(tokens: list[str], k: int) -> list[tuple]:
  counts = Counter(tokens)
  return heapq.nlargest(k, counts.items(), key=lambda x: x[1])

top_k_tokens(tokens, 4)

[('the', 3), ('cat', 2), ('sat', 1), ('on', 1)]

In [8]:
# data structure que 2
# design context manager that keeps the sliding window of the messages within the max token limit
from collections import deque

class ContextManager:
  def __init__(self, max_tokens: int, system_prompt: str):
    self.max_tokens = max_tokens
    self.system_prompt = system_prompt
    self._messages: deque = deque()
    self._token_count = len(system_prompt.split())

  def _approx_tokens(self, msg: dict) -> int:
    return len(msg["content"].split()) + 4

  def add_message(self, message: dict):
    t = self._approx_tokens(message)
    while self._token_count + t > self.max_tokens and self._messages:
      old = self._messages.popleft()
      self._token_count -= self._approx_tokens(old)
    self._messages.append(message)
    self._token_count += t

  def get_messages(self) -> list[dict]:
    return list(self._messages)

In [11]:
# deduplication function for the RAG chunks
chunks = [
    {"source": "10K_2024.pdf", "score": 0.91, "text": "Revenue grew..."},
    {"source": "10K_2024.pdf", "score": 0.74, "text": "Operating costs..."},
    {"source": "earnings_Q1.pdf", "score": 0.88, "text": "EPS beat..."},
]
# Return: one chunk per source, highest score wins

def dedup_chunks(chunks: list[dict]) -> list[dict]:
  best = {}
  for c in chunks:
    src = c["source"]
    if src not in best or c["score"] > best[src]["score"]:
      best[src] = c
  return list(best.values())

In [12]:
dedup_chunks(chunks)

[{'source': '10K_2024.pdf', 'score': 0.91, 'text': 'Revenue grew...'},
 {'source': 'earnings_Q1.pdf', 'score': 0.88, 'text': 'EPS beat...'}]

In [16]:
# genai specific data validation models

from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum

class Sentiment(str, Enum):
  positive = "positive"
  negative = "negative"
  neutral = "neutral"

class ComplianceFlag(BaseModel):
  severity: str
  description: str

class ResearchReport(BaseModel):
  ticker: str = Field(..., pattern=r"^[A-Z]{1,5}$")
  summary: str = Field(..., min_length=10)
  sentiment: Sentiment
  confidence: float
  citations: list[str]
  flags: Optional[List[ComplianceFlag]] = None
  pe_ratio: Optional[float] = None

  @field_validator("citations")
  @classmethod
  def must_have_sources(cls, v):
        if len(v) == 0:
            raise ValueError("Report must cite at least one source")
        return v

In [17]:
# python generator for the cerebras streaming

from typing import Generator

def stream_llm(prompt: str) -> Generator[str, None, None]:
  stream = client.chat.compleations.create(
      model="llama-4-scout-17b-16e-instruct",
      messages=[
          {"role": "system", "content": "You are a helpful assistant."},
          {"role": "user", "content": prompt},
      ],
      stream=True,
      max_tokens=1024,
  )
  for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
      yield delta


async def async_stream(prompt: str):
  for token in stream_llm(prompt):
    yield token
    await asyncio.sleep(0.3)